# 01 · Preprocesamiento

Objetivo: dejar dos tablas listas para el análisis exploratorio y el modelo.

- `videos_maestra`: una fila por vídeo, con metadatos, totales y acumulados de los primeros días.
- `panel_diario`: una fila por vídeo y día, contando los días desde el primer día con visualizaciones.

Decisiones principales:
1. Se analizan los vídeos públicos con datos de Analytics. Los directos quedan fuera.
2. Los vídeos largos de 180 s o menos se mantienen como largos y se marcan.
3. El día 0 de cada vídeo es su primer día con visualizaciones.
4. Se registra si la ventana de cada vídeo cruza el cambio de criterio de visualizaciones del 24/08/2026.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

CSV_DIR = Path("../data/csv")
SALIDA_DIR = Path("../data/processed")
SALIDA_DIR.mkdir(parents=True, exist_ok=True)

ZONA_HORARIA = "Europe/Madrid"
HORIZONTES = (1, 2, 3, 7, 28)

# Desde esta fecha YouTube cuenta como visualización la reproducción desde el primer fotograma
FECHA_CAMBIO_VISUALIZACION = pd.Timestamp("2026-08-24")

# Duración máxima de un Short: 60 s hasta el 14/10/2024 y 180 s desde el 15/10/2024.
# YouTube clasifica además por formato (vertical o cuadrado), dato que no consta aquí.
FECHA_CAMBIO_LIMITE_SHORT = pd.Timestamp("2024-10-15", tz="UTC")
LIMITE_SHORT_ANTES_S = 60
LIMITE_SHORT_DESPUES_S = 180


def leer(nombre: str, **kwargs) -> pd.DataFrame:
    """Lee un CSV de la carpeta de datos (UTF-8 con posible marca BOM)."""
    return pd.read_csv(CSV_DIR / nombre, encoding="utf-8-sig", **kwargs)


perf = leer("video_performance.csv")
perf["published_at"] = pd.to_datetime(perf["published_at"], utc=True)

diario = leer("analytics_video_daily.csv", parse_dates=["day"])
ingresos_diario = leer("analytics_video_revenue_daily.csv", parse_dates=["day"])
canal_diario = leer("analytics_channel_daily.csv", parse_dates=["day"])
tipo_diario = leer("analytics_channel_content_type_daily.csv", parse_dates=["day"])
suscripcion = leer("analytics_video_subscribed_status.csv")

FIN_ANALYTICS = canal_diario["day"].max()
print(f"Último día con datos de Analytics: {FIN_ANALYTICS.date()}")
print(f"Vídeos en video_performance: {len(perf)}")

Último día con datos de Analytics: 2026-10-20
Vídeos en video_performance: 357


## 1. Población de análisis

Se parte de los vídeos del fichero y se conservan solo los públicos (la entrega ya los
trae filtrados; la comprobación protege si se ejecuta con ficheros sin filtrar).
Después se descartan los que no tienen datos de Analytics (publicados después del último
día descargado) y los directos, porque el informe oficial de YouTube los cuenta como
vídeo bajo demanda y sus cifras no son comparables.

In [2]:
pasos = [("Vídeos en el fichero", len(perf))]

poblacion = perf[perf["privacy_status"] == "public"].copy()
pasos.append(("Públicos", len(poblacion)))

poblacion = poblacion[poblacion["views"].notna()].copy()
pasos.append(("Con datos de Analytics", len(poblacion)))

poblacion = poblacion[poblacion["format"] != "live"].copy()
pasos.append(("Sin directos", len(poblacion)))

print(pd.DataFrame(pasos, columns=["paso", "vídeos"]).to_string(index=False))
print()
print(poblacion["format"].value_counts().to_string())

# Vídeos largos con una duración que cabría en un Short: se mantienen como largos
limite_short = np.where(
    poblacion["published_at"] >= FECHA_CAMBIO_LIMITE_SHORT,
    LIMITE_SHORT_DESPUES_S,
    LIMITE_SHORT_ANTES_S,
)
poblacion["largo_breve"] = (poblacion["format"] == "long") & (
    poblacion["duration_seconds"] <= limite_short
)
print(f"\nVídeos largos marcados como breves: {int(poblacion['largo_breve'].sum())}")

                  paso  vídeos
  Vídeos en el fichero     357
              Públicos     357
Con datos de Analytics     355
          Sin directos     352

format
long     260
short     92

Vídeos largos marcados como breves: 7


## 2. Hora y día de publicación

`published_at` está en UTC. Se convierte a hora de Madrid, que es la de la audiencia, para
que el análisis de la hora de publicación no salga desplazado una o dos horas.

In [3]:
publicado_local = poblacion["published_at"].dt.tz_convert(ZONA_HORARIA)
poblacion["hora_madrid"] = publicado_local.dt.hour
poblacion["dia_semana"] = publicado_local.dt.dayofweek  # 0 = lunes
poblacion["fecha_publicacion_utc"] = (
    poblacion["published_at"].dt.tz_convert("UTC").dt.tz_localize(None).dt.normalize()
)

print(poblacion.groupby("dia_semana").size().to_string())

dia_semana
0    55
1    62
2    53
3    57
4    54
5    46
6    25


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15824\1111682050.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  poblacion["dia_semana"] = publicado_local.dt.dayofweek  # 0 = lunes


## 3. Panel diario y día 0

La serie diaria empieza el día anterior a la publicación, así que no se puede suponer que
la fecha de publicación coincida con el primer día de datos (el día de Analytics puede
estar en otra zona horaria). Por eso el día 0 de cada vídeo es su primer día con
visualizaciones, y se comprueba cuánto se aleja de la fecha de publicación.

In [4]:
ids = set(poblacion["video_id"])
panel = diario[diario["video_id"].isin(ids)].copy()
panel["subs_netos"] = panel["subscribersGained"] - panel["subscribersLost"]

primer_dia = (
    panel[panel["views"] > 0].groupby("video_id")["day"].min().rename("primer_dia")
)
sin_visualizaciones = ids - set(primer_dia.index)
print(f"Vídeos sin ningún día con visualizaciones: {len(sin_visualizaciones)}")

panel = panel.merge(primer_dia, on="video_id", how="inner")
panel["dia_rel"] = (panel["day"] - panel["primer_dia"]).dt.days
panel = panel[panel["dia_rel"] >= 0].sort_values(["video_id", "dia_rel"])

# Desfase entre el primer día con visualizaciones y la fecha de publicación en UTC
desfase = primer_dia.to_frame().join(
    poblacion.set_index("video_id")["fecha_publicacion_utc"]
)
desfase = (desfase["primer_dia"] - desfase["fecha_publicacion_utc"]).dt.days
print("\nDesfase en días (primer día con vistas - fecha de publicación UTC):")
print(desfase.value_counts().sort_index().to_string())

# Días sin fila dentro del rango de cada vídeo (suman cero, no afectan a los acumulados)
esperados = (FIN_ANALYTICS - primer_dia).dt.days + 1
presentes = panel.groupby("video_id").size()
huecos = (esperados - presentes.reindex(esperados.index)).clip(lower=0)
print(f"\nVídeos con días sin fila: {int((huecos > 0).sum())} | días totales: {int(huecos.sum())}")

Vídeos sin ningún día con visualizaciones: 0

Desfase en días (primer día con vistas - fecha de publicación UTC):
-1     39
 0    313

Vídeos con días sin fila: 0 | días totales: 0


In [7]:
revision = poblacion.set_index("video_id")[["published_at"]].join(desfase.rename("desfase"))
revision["hora_utc"] = revision["published_at"].dt.hour
print(revision.groupby("desfase")["hora_utc"].agg(["min", "max", "count"]).to_string())

         min  max  count
desfase                 
-1         4   18     39
 0         7   23    313


## 4. Acumulados de los primeros días

Para cada vídeo se suma lo ocurrido en sus primeros 1, 2, 3, 7 y 28 días (día 0 incluido).
Un día equivale a unas 24 h, y dos días a unas 48 h. Si el vídeo no tiene todos los días
de la ventana en los datos, el acumulado queda vacío en vez de infravalorado.

In [5]:
METRICAS_TEMPRANAS = {
    "views": "vistas",
    "estimatedMinutesWatched": "minutos",
    "likes": "likes",
    "comments": "comentarios",
    "shares": "compartidos",
    "subs_netos": "subs_netos",
}

bloques = []
for horizonte in HORIZONTES:
    suma = (
        panel[panel["dia_rel"] < horizonte]
        .groupby("video_id")[list(METRICAS_TEMPRANAS)]
        .sum()
    )
    suma.columns = [f"{alias}_{horizonte}d" for alias in METRICAS_TEMPRANAS.values()]
    bloques.append(suma)
acumulados = pd.concat(bloques, axis=1).reset_index()

# Ingresos de los primeros 28 días, con el mismo criterio de día 0
ingresos_panel = ingresos_diario[ingresos_diario["video_id"].isin(ids)].merge(
    primer_dia, on="video_id", how="inner"
)
ingresos_panel["dia_rel"] = (ingresos_panel["day"] - ingresos_panel["primer_dia"]).dt.days
ingresos_28d = (
    ingresos_panel[ingresos_panel["dia_rel"].between(0, 27)]
    .groupby("video_id")["estimatedRevenue"]
    .sum()
    .rename("ingresos_28d")
    .reset_index()
)

## 5. Tabla maestra

Se une todo en una fila por vídeo. `dias_observados` indica cuántos días de recorrido hay
en los datos y decide con qué horizontes se puede usar cada vídeo.

In [6]:
RENOMBRAR = {
    "format": "formato",
    "views": "vistas_total",
    "estimatedMinutesWatched": "minutos_total",
    "averageViewDuration": "duracion_media_vista_s",
    "averageViewPercentage": "porcentaje_visto_medio",
    "subscribersGained": "subs_ganados_total",
    "subscribersLost": "subs_perdidos_total",
    "likes": "likes_total",
    "comments": "comentarios_total",
    "shares": "compartidos_total",
    "estimatedRevenue": "ingresos_total",
}
COLUMNAS_BASE = [
    "video_id", "title", "published_at", "fecha_publicacion_utc", "hora_madrid",
    "dia_semana", "duration_seconds", "format", "largo_breve", "title_length",
    "description_length", "tags_count", "has_captions", "views",
    "estimatedMinutesWatched", "averageViewDuration", "averageViewPercentage",
    "subscribersGained", "subscribersLost", "likes", "comments", "shares",
    "estimatedRevenue",
]

maestra = (
    poblacion[COLUMNAS_BASE]
    .rename(columns=RENOMBRAR)
    .merge(primer_dia.reset_index(), on="video_id", how="inner")
    .merge(acumulados, on="video_id", how="left")
    .merge(ingresos_28d, on="video_id", how="left")
)
maestra["dias_observados"] = (FIN_ANALYTICS - maestra["primer_dia"]).dt.days + 1
maestra["rpm_total"] = (
    maestra["ingresos_total"] / maestra["vistas_total"].where(maestra["vistas_total"] > 0) * 1000
)

# Los acumulados de una ventana incompleta se anulan
for horizonte in HORIZONTES:
    incompleta = maestra["dias_observados"] < horizonte
    columnas = [c for c in maestra.columns if c.endswith(f"_{horizonte}d")]
    maestra.loc[incompleta, columnas] = np.nan

print(f"Vídeos en la tabla maestra: {len(maestra)}")
print(maestra["formato"].value_counts().to_string())

Vídeos en la tabla maestra: 352
formato
long     260
short     92


## 6. Cobertura por horizonte

Cuántos vídeos tienen la ventana completa en cada horizonte. Es el tamaño real de la
muestra con la que se podrá entrenar y validar cada modelo.

In [8]:
cobertura = pd.DataFrame(
    {
        f"{h} d": maestra.groupby("formato")["dias_observados"].apply(
            lambda s, h=h: int((s >= h).sum())
        )
        for h in HORIZONTES
    }
)
print(cobertura.to_string())

         1 d  2 d  3 d  7 d  28 d
formato                          
long     260  259  258  256   246
short     92   92   92   92    92


## 7. Cambio en el criterio de visualizaciones (24/08/2026)

Desde esa fecha YouTube cuenta como visualización la reproducción desde el primer
fotograma. Para ver si afecta a los datos descargados se compara, en los vídeos largos
(`videoOnDemand`), la evolución semanal de visualizaciones, minutos y minutos por
visualización. Si las visualizaciones suben y los minutos por visualización bajan sin que
cambien los minutos, la serie de visualizaciones ya usa el criterio nuevo.
Es una comparación orientativa: el contenido publicado cada semana también varía.

In [9]:
largos_canal = (
    tipo_diario[tipo_diario["creatorContentType"] == "videoOnDemand"]
    .set_index("day")
    .sort_index()
)

inicio = FECHA_CAMBIO_VISUALIZACION - pd.Timedelta(days=56)
semanal = largos_canal.loc[inicio:, ["views", "estimatedMinutesWatched"]].resample("W-SUN").sum()
semanal["min_por_vista"] = semanal["estimatedMinutesWatched"] / semanal["views"]
print("Vídeos largos del canal, por semana (cada fila termina en domingo):")
print(semanal.round(2).to_string())

antes = largos_canal.loc[
    FECHA_CAMBIO_VISUALIZACION - pd.Timedelta(days=28): FECHA_CAMBIO_VISUALIZACION - pd.Timedelta(days=1)
]
despues = largos_canal.loc[
    FECHA_CAMBIO_VISUALIZACION: FECHA_CAMBIO_VISUALIZACION + pd.Timedelta(days=27)
]
comparacion = pd.DataFrame(
    {
        "4 semanas antes": [
            antes["views"].mean(),
            antes["estimatedMinutesWatched"].mean(),
            antes["estimatedMinutesWatched"].sum() / antes["views"].sum(),
        ],
        "4 semanas después": [
            despues["views"].mean(),
            despues["estimatedMinutesWatched"].mean(),
            despues["estimatedMinutesWatched"].sum() / despues["views"].sum(),
        ],
    },
    index=["visualizaciones/día", "minutos/día", "minutos por visualización"],
)
comparacion["variación_%"] = (comparacion["4 semanas después"] / comparacion["4 semanas antes"] - 1) * 100
print("\nComparación de medias diarias:")
print(comparacion.round(2).to_string())

Vídeos largos del canal, por semana (cada fila termina en domingo):
             views  estimatedMinutesWatched  min_por_vista
day                                                       
2026-07-05   29915                   208601           6.97
2026-07-12   19506                    91637           4.70
2026-07-19   14273                    57563           4.03
2026-07-26   10301                    41831           4.06
2026-08-02   18926                    97600           5.16
2026-08-09   13702                    78581           5.74
2026-08-16   11636                    52149           4.48
2026-08-23   39042                   172824           4.43
2026-08-30   29520                    85771           2.91
2026-09-06   73221                   328266           4.48
2026-09-13  209017                   400798           1.92
2026-09-20  192428                   316548           1.65
2026-09-27  158387                   276146           1.74
2026-10-04   82232                   158836    

Con el resultado anterior se marca cada vídeo según si su ventana de 7 y de 28 días queda
antes, después o a caballo del 24/08/2026. El modelo podrá excluir o tratar aparte los
vídeos con ventana mixta.

In [10]:
def clasificar_periodo(primer_dia_video: pd.Series, dias: int) -> pd.Series:
    """Indica si la ventana de `dias` días queda antes, después o a caballo del cambio."""
    ultimo_dia = primer_dia_video + pd.to_timedelta(dias - 1, unit="D")
    return pd.Series(
        np.select(
            [ultimo_dia < FECHA_CAMBIO_VISUALIZACION, primer_dia_video >= FECHA_CAMBIO_VISUALIZACION],
            ["previo", "posterior"],
            default="mixto",
        ),
        index=primer_dia_video.index,
    )


maestra["periodo_7d"] = clasificar_periodo(maestra["primer_dia"], 7)
maestra["periodo_28d"] = clasificar_periodo(maestra["primer_dia"], 28)
print(maestra.groupby(["formato", "periodo_28d"]).size().unstack(fill_value=0).to_string())

periodo_28d  mixto  posterior  previo
formato                              
long             9         30     221
short            6          1      85


## 8. Comprobaciones de coherencia

- Los acumulados del panel deben coincidir con los totales de cada vídeo.
- Todos los vídeos deben tener las dos categorías de audiencia (suscrita y no suscrita)
  en el informe correspondiente.
- No debe haber duplicados.

In [11]:
# Panel frente a totales por vídeo
suma_panel = panel.groupby("video_id")[["views", "estimatedMinutesWatched"]].sum()
comparacion_totales = maestra.set_index("video_id")[["vistas_total", "minutos_total"]].join(suma_panel)
dif_vistas = (comparacion_totales["vistas_total"] - comparacion_totales["views"]).abs()
dif_minutos = (comparacion_totales["minutos_total"] - comparacion_totales["estimatedMinutesWatched"]).abs()
print(f"Vídeos con diferencia de visualizaciones (panel vs total): {int((dif_vistas > 0).sum())}")
print(f"Vídeos con diferencia de minutos (panel vs total):        {int((dif_minutos > 0).sum())}")

# Audiencia suscrita y no suscrita
suscripcion = suscripcion[suscripcion["video_id"].isin(ids)]
print("\nValores de subscribedStatus:", sorted(suscripcion["subscribedStatus"].unique()))
categorias = suscripcion.groupby("video_id")["subscribedStatus"].nunique()
print(f"Vídeos con una sola categoría: {int((categorias < 2).sum())}")
print(f"Vídeos de la tabla maestra sin este informe: {len(set(maestra['video_id']) - set(categorias.index))}")

# Duplicados y nulos
print(f"\nvideo_id duplicados: {int(maestra['video_id'].duplicated().sum())}")
nulos = maestra.isna().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0].to_string())

Vídeos con diferencia de visualizaciones (panel vs total): 0
Vídeos con diferencia de minutos (panel vs total):        351

Valores de subscribedStatus: ['SUBSCRIBED', 'UNSUBSCRIBED']
Vídeos con una sola categoría: 0
Vídeos de la tabla maestra sin este informe: 0

video_id duplicados: 0
Columnas con nulos:
vistas_2d           1
minutos_2d          1
likes_2d            1
comentarios_2d      1
compartidos_2d      1
subs_netos_2d       1
vistas_3d           2
minutos_3d          2
likes_3d            2
comentarios_3d      2
compartidos_3d      2
subs_netos_3d       2
vistas_7d           4
minutos_7d          4
likes_7d            4
comentarios_7d      4
compartidos_7d      4
subs_netos_7d       4
vistas_28d         14
minutos_28d        14
likes_28d          14
comentarios_28d    14
compartidos_28d    14
subs_netos_28d     14
ingresos_28d       14


## 9. Guardado

Las tablas procesadas se leerán en el análisis exploratorio y en el modelo.

In [12]:
panel_salida = panel[
    ["video_id", "day", "dia_rel", "views", "estimatedMinutesWatched", "likes",
     "comments", "shares", "subs_netos"]
]
maestra.to_csv(SALIDA_DIR / "videos_maestra.csv", index=False)
panel_salida.to_csv(SALIDA_DIR / "panel_diario.csv", index=False)
print(f"videos_maestra.csv: {len(maestra)} filas | panel_diario.csv: {len(panel_salida)} filas")

videos_maestra.csv: 352 filas | panel_diario.csv: 128524 filas
